# PINN — Newton's Law of Cooling

This notebook introduces **Physics-Informed Neural Networks (PINNs)** on the simplest
possible ODE: Newton's law of cooling.

**Three models compared:**

| Model | Loss |
|-------|------|
| **NN** | data only |
| **PINN** | data + ODE residual (R known) |
| **PINN-ID** | data + ODE residual (R learnable) |

**Key insight:** The NN overfits the 10 noisy observations and diverges far from
equilibrium in extrapolation. The PINN correctly decays to $T_\text{env}$ because
the ODE is enforced at collocation points spread over the *full* evaluation domain,
not just where data exist. PINN-ID simultaneously fits the trajectory and identifies
the unknown cooling rate from noisy data alone.


In [ ]:
import os, sys
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
%matplotlib inline

sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))

SEED = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
print(f"Device: {DEVICE}")

RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
COLORS = {"True": "#2c3e50", "NN": "#e74c3c", "PINN": "#3498db", "PINN-ID": "#9b59b6"}

## The System: Newton's Law of Cooling

The object exchanges heat with its environment according to:

$$\frac{dT}{dt} = R\,(T_{\text{env}} - T)$$

with exact solution $T(t) = T_{\text{env}} + (T_0 - T_{\text{env}})\,e^{-Rt}$.

**Setup:**
- $T_{\text{env}} = 25\,°\text{C}$, $T_0 = 100\,°\text{C}$, $R = 0.005\,\text{s}^{-1}$
- **10 noisy observations** ($\sigma = 2\,°\text{C}$) from $t \in [0, 300\,\text{s}]$
- Evaluation domain: $[0, 1000\,\text{s}]$ — the model must extrapolate far beyond the data


In [ ]:
T_ENV, T0, R = 25.0, 100.0, 0.005

def cooling_law(t):
    return T_ENV + (T0 - T_ENV) * np.exp(-R * t)

np.random.seed(10)
times  = np.linspace(0, 1000, 1000)
temps  = cooling_law(times)
t_data = np.linspace(0, 300, 10)
T_data = cooling_law(t_data) + 2.0 * np.random.randn(10)

# Quick look at the data
plt.figure(figsize=(10, 3))
plt.plot(times, temps, color=COLORS["True"], lw=1.5, ls="--", label="Ground truth")
plt.scatter(t_data, T_data, s=60, color=COLORS["True"], zorder=5, label="10 noisy obs")
plt.axvline(300, color="gray", ls=":", lw=1.2, label="train end")
plt.axvspan(300, times[-1], alpha=0.06, color="gray")
plt.xlabel("Time (s)"); plt.ylabel("T (°C)"); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## Network Architecture

All three models share the same network: **4 hidden layers of 100 ReLU units**,
mapping $t \mapsto T(t)$.

The `Net` class accepts an optional `loss2` callable — the physics residual function.
If provided, it is added to the MSE data loss with weight `loss2_weight`.

`NetDiscovery` extends `Net` by adding `self.r = nn.Parameter(...)`, making the
cooling rate a learnable variable. The ODE residual then provides the gradient signal
that drives `r` towards the true value — no additional supervision needed.


In [ ]:
def np_to_th(x):
    return torch.from_numpy(x).float().to(DEVICE).reshape(len(x), -1)

def grad(outputs, inputs):
    return torch.autograd.grad(
        outputs, inputs, grad_outputs=torch.ones_like(outputs), create_graph=True)


class Net(nn.Module):
    def __init__(self, input_dim, output_dim, n_units=100,
                 epochs=1000, lr=1e-3, loss2=None, loss2_weight=0.1):
        super().__init__()
        self.epochs, self.loss2, self.loss2_weight, self.lr = epochs, loss2, loss2_weight, lr
        self.layers = nn.Sequential(
            nn.Linear(input_dim, n_units), nn.ReLU(),
            nn.Linear(n_units,   n_units), nn.ReLU(),
            nn.Linear(n_units,   n_units), nn.ReLU(),
            nn.Linear(n_units,   n_units), nn.ReLU(),
        )
        self.out = nn.Linear(n_units, output_dim)

    def forward(self, x):
        return self.out(self.layers(x))

    def fit(self, t_np, T_np, log_every=5000):
        Xt, yt = np_to_th(t_np), np_to_th(T_np)
        opt = optim.Adam(self.parameters(), lr=self.lr)
        losses = []
        self.train()
        for ep in range(1, self.epochs + 1):
            opt.zero_grad()
            loss = nn.MSELoss()(self.forward(Xt), yt)
            if self.loss2 is not None:
                loss = loss + self.loss2_weight * self.loss2(self)
            loss.backward(); opt.step()
            losses.append(loss.item())
            if ep % log_every == 0:
                print(f"  ep {ep:6d}/{self.epochs}  loss={losses[-1]:.4e}")
        return losses

    def predict(self, t_np):
        self.eval()
        with torch.no_grad():
            return self.forward(np_to_th(t_np)).cpu().numpy().squeeze()


class NetDiscovery(Net):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.r = nn.Parameter(torch.tensor([0.0]))   # learnable rate, starts at 0

    def fit(self, t_np, T_np, log_every=5000):
        Xt, yt = np_to_th(t_np), np_to_th(T_np)
        opt = optim.Adam(self.parameters(), lr=self.lr)
        losses, r_history = [], []
        self.train()
        for ep in range(1, self.epochs + 1):
            opt.zero_grad()
            loss = nn.MSELoss()(self.forward(Xt), yt)
            if self.loss2 is not None:
                loss = loss + self.loss2_weight * self.loss2(self)
            loss.backward(); opt.step()
            losses.append(loss.item()); r_history.append(self.r.item())
            if ep % log_every == 0:
                print(f"  ep {ep:6d}/{self.epochs}  loss={losses[-1]:.4e}  r={self.r.item():.5f}")
        return losses, r_history

## Physics Loss: the ODE Residual

Rearranging the ODE to residual form:

$$\mathcal{R}[T_\theta](t) = R\,(T_{\text{env}} - T_\theta(t)) - \frac{dT_\theta}{dt} = 0$$

The time derivative $dT_\theta/dt$ is computed via **automatic differentiation** —
PyTorch differentiates through the network graph exactly. The loss penalises the
squared residual at 1000 collocation points sampled over $[0, 1000\,\text{s}]$:

$$\mathcal{L}_{\text{phys}} = \frac{1}{M}\sum_{j=1}^{M} \mathcal{R}[T_\theta](t_j)^2$$

For PINN-ID, the known $R$ is replaced by `model.r`, so the same residual gradient
also updates the identified parameter.


In [ ]:
def physics_loss(model):
    ts = torch.linspace(0, 1000, 1000).view(-1, 1).requires_grad_(True).to(DEVICE)
    T_pred = model(ts)
    dT = grad(T_pred, ts)[0]
    return torch.mean((R * (T_ENV - T_pred) - dT)**2)

def physics_loss_discovery(model):
    ts = torch.linspace(0, 1000, 1000).view(-1, 1).requires_grad_(True).to(DEVICE)
    T_pred = model(ts)
    dT = grad(T_pred, ts)[0]
    return torch.mean((model.r * (T_ENV - T_pred) - dT)**2)

## Training

Each model is trained for a different number of epochs:
- **NN**: 20 000 epochs, lr=1e-5 (data only)
- **PINN**: 30 000 epochs, lr=1e-5 (data + physics, known R)
- **PINN-ID**: 40 000 epochs, lr=5e-6 (data + physics, R learnable)

The extra epochs for PINN-ID give the parameter identification time to converge.
Training takes ~30–90 s per model on CPU.


In [ ]:
print("Training NN (data only) ...")
net = Net(1, 1, loss2=None, epochs=20_000, lr=1e-5).to(DEVICE)
net.fit(t_data, T_data)

In [ ]:
print("Training PINN (known R) ...")
pinn = Net(1, 1, loss2=physics_loss, loss2_weight=1, epochs=30_000, lr=1e-5).to(DEVICE)
pinn.fit(t_data, T_data)

In [ ]:
print("Training PINN-ID (learnable R, initialised at 0) ...")
pinn_id = NetDiscovery(1, 1, loss2=physics_loss_discovery,
                       loss2_weight=1, epochs=40_000, lr=5e-6).to(DEVICE)
_, r_history = pinn_id.fit(t_data, T_data)

R_id = pinn_id.r.item()
print(f"\nIdentified  R = {R_id:.5f}")
print(f"True        R = {R:.5f}")
print(f"Relative error: {abs(R_id - R)/R * 100:.2f}%")

## Results

We report interpolation and extrapolation RMSE separately to distinguish in-distribution
accuracy from generalisation. The decisive metric is **extrapolation RMSE** — how well
each model reconstructs the cooling curve beyond $t = 300\,\text{s}$.


In [ ]:
T_nn   = net.predict(times)
T_pinn = pinn.predict(times)
T_id   = pinn_id.predict(times)

split = 300.0
in_m, out_m = times <= split, times > split

def rmse(pred, mask):
    return float(np.sqrt(np.mean((pred[mask] - temps[mask])**2)))

print(f"{'Model':<10}  {'Interp RMSE':>12}  {'Extrap RMSE':>12}")
print("-" * 40)
for name, pred in [("NN", T_nn), ("PINN", T_pinn), ("PINN-ID", T_id)]:
    print(f"{name:<10}  {rmse(pred, in_m):>12.3f}  {rmse(pred, out_m):>12.3f}")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 11))
fig.suptitle(
    f"PINN — Newton's Law of Cooling\n"
    f"T_env={T_ENV} °C,  T0={T0} °C,  R={R} s⁻¹  |  10 noisy obs from [0,300 s]",
    fontsize=12, fontweight="bold")

# Trajectories
ax = axes[0]
ax.plot(times, temps, color=COLORS["True"], lw=1, ls="--", label="Ground truth", zorder=5)
ax.plot(times, T_nn,   color=COLORS["NN"],      lw=1.8, label="NN (data only)")
ax.plot(times, T_pinn, color=COLORS["PINN"],    lw=1.8, label=f"PINN (R={R})")
ax.plot(times, T_id,   color=COLORS["PINN-ID"], lw=1.8, label=f"PINN-ID (R_id={R_id:.4f})")
ax.scatter(t_data, T_data, s=60, color=COLORS["True"], zorder=10,
           edgecolors="white", linewidths=0.5, label="noisy obs")
ax.axvline(split, color="gray", ls=":", lw=1.4, label="train end")
ax.axvspan(split, times[-1], alpha=0.07, color="gray")
ax.set_ylabel("Temperature (°C)", fontsize=11); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
ax.set_title("Trajectory", fontsize=11, fontweight="bold")

# Absolute error (log scale)
ax = axes[1]
for name, pred, c in [("NN", T_nn, COLORS["NN"]),
                       ("PINN", T_pinn, COLORS["PINN"]),
                       ("PINN-ID", T_id, COLORS["PINN-ID"])]:
    ax.semilogy(times, np.abs(pred - temps) + 1e-3, color=c, lw=1.6, label=name)
ax.axvline(split, color="gray", ls=":", lw=1.4)
ax.axvspan(split, times[-1], alpha=0.07, color="gray")
ax.set_ylabel("|T error| °C  (log)", fontsize=11); ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# R convergence
ax = axes[2]
ax.plot(np.arange(1, len(r_history)+1), r_history, color=COLORS["PINN-ID"], lw=1.5, label="R identified")
ax.axhline(R,   color=COLORS["True"], lw=1.2, ls="--", label=f"R true = {R}")
ax.axhline(0.0, color="gray",         lw=1.0, ls=":",  label="R init = 0")
ax.set_xlabel("Epoch", fontsize=11); ax.set_ylabel("R (s⁻¹)", fontsize=11)
ax.set_title("Cooling rate identification", fontsize=11, fontweight="bold")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "pinn_cooling.png"), dpi=150, bbox_inches="tight")
plt.show()